In [ ]:
# A100, H100 환경을 위한 Flash Attention 2 설치 (처음 1회만 실행)
!pip install flash-attn --no-build-isolation

import os, re, math, random
import pandas as pd
from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader
import torch
from torchvision import transforms
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
# 전체 데이터 사용 (기존 sample(n=200) 제한 해제)
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

# --- [추가됨] 이미지 전처리 및 증강 기법 ---
def enhance_image(image):
    image = ImageEnhance.Contrast(image).enhance(1.2)
    image = ImageEnhance.Sharpness(image).enhance(1.5)
    return image

train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomRotation(degrees=3), 
])

# 모델 지시사항 (기존 동일)
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        
        # --- [추가됨] 화질 개선 및 증강 적용 ---
        img = enhance_image(img)
        if self.train:
            img = train_transform(img)

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})
            
        return {"messages": messages, "image": img}

class DataCollator:
    def __init__(self, processor, train=True):
        self.processor = processor
        self.train = train

    def __call__(self, samples):
        texts, images = [], []
        for sample in samples:
            messages = sample["messages"]
            img = sample["image"]
            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=not self.train
            )
            texts.append(text)
            images.append(img)
            
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        if self.train:
            enc["labels"] = enc["input_ids"].clone()
        return enc

processor = AutoProcessor.from_pretrained(MODEL_ID)

split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# A100 VRAM을 넉넉히 활용하기 위해 batch_size 상향 가능 (OOM 시 1~2로 조정)
BATCH_SIZE = 4 
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

In [ ]:
# --- [수정됨] Bfloat16 및 Flash Attention 2 적용 (양자화 불필요) ---
print("모델 로딩 중... (Flash Attention 2 및 Bfloat16 적용)")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto"
)
base_model.gradient_checkpointing_enable()

# --- [수정됨] LoRA Rank, Alpha 상향 조정 ---
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.to(device)

GRAD_ACCUM = 4
EPOCHS = 3 # 전체 데이터이므로 2~3회 반복 권장

# 옵티마이저 및 코사인 스케줄러 적용
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
num_training_steps = EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(num_training_steps*0.1), num_training_steps=num_training_steps)

scaler = torch.cuda.amp.GradScaler(enabled=True)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        avg_loss = running / min(step, GRAD_ACCUM) if step < GRAD_ACCUM else running / GRAD_ACCUM
        progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
        if step % GRAD_ACCUM == 0: running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
        print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")

# --- [저장] 기존 경로 및 방식 완벽 유지 ---
SAVE_DIR = "/content/qwen2_5_vl_3b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines: return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]: return last
    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]: return tok
    return "a"

model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    # 전처리는 가독성을 위해 추가(추론시에도 유리함)
    img = enhance_image(img) 
    
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
        output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
        preds.append(extract_choice(output_text))

# --- [저장] 기존 경로 및 방식 완벽 유지 ---
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")